In [1]:
# Step 1: Mount Drive, install/import everything needed

from google.colab import drive
drive.mount('/content/drive')

base = "/content/drive/MyDrive/Quantum_DL_MNIST"

!pip install pennylane pennylane-lightning -q

import torch
import torch.nn as nn
import numpy as np
import pennylane as qml
import json, copy, time, os, random, sys
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

sys.path.append(f"{base}/notebooks")
from shared_backbone import CNNBackbone

print("Everything imported, PennyLane version:", qml.__version__)

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 11.6 MB/s eta 0:00:00
Everything imported, PennyLane version: 0.45.1


In [2]:
# Step 2: Load data (same as always)

train_idx = np.load(f"{base}/splits/train_indices.npy")
val_idx   = np.load(f"{base}/splits/val_indices.npy")
test_idx  = np.load(f"{base}/splits/test_indices.npy")

with open(f"{base}/splits/normalization_stats.json") as f:
    norm_stats = json.load(f)
mean, std = norm_stats["mean"], norm_stats["std"]

train_full = torchvision.datasets.MNIST(root=f"{base}/data", train=True, download=True, transform=transforms.ToTensor())
test_full  = torchvision.datasets.MNIST(root=f"{base}/data", train=False, download=True, transform=transforms.ToTensor())

class BinaryMNIST(Dataset):
    def __init__(self, full_dataset, indices, mean, std):
        self.full_dataset, self.indices, self.mean, self.std = full_dataset, indices, mean, std
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        real_idx = self.indices[i]
        image, label = self.full_dataset[real_idx]
        image = (image - self.mean) / self.std
        return image, torch.tensor(float(label))

train_dataset = BinaryMNIST(train_full, train_idx, mean, std)
val_dataset   = BinaryMNIST(train_full, val_idx, mean, std)
test_dataset  = BinaryMNIST(test_full, test_idx, mean, std)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Data ready:", len(train_loader), len(val_loader), len(test_loader), "batches")

Data ready: 317 80 67 batches


In [9]:
# Step 3 (fixed): Correct feature indexing for batched execution
# BUG FOUND: inputs[i] was grabbing SAMPLE i, not FEATURE i.
# Fixed to inputs[..., i], which correctly selects feature i across
# the whole batch, enabling proper vectorized execution.

n_qubits = 4
n_layers = 2

dev = qml.device("default.qubit", wires=n_qubits)
backend_used = "default.qubit"

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)   # FIXED: was inputs[i]

    for layer in range(n_layers):
        for i in range(n_qubits):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(n_qubits):
            qml.CNOT(wires=[i, (i + 1) % n_qubits])

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

print("Quantum circuit redefined with corrected batch indexing")

Quantum circuit redefined with corrected batch indexing


In [10]:
# Step 4 (re-run): Recreate the TorchLayer

weight_shapes = {"weights": (n_layers, n_qubits, 2)}
quantum_layer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

print("TorchLayer created")
print("Quantum layer parameters:", sum(p.numel() for p in quantum_layer.parameters()))

TorchLayer created
Quantum layer parameters: 16


In [11]:
# Step 5 (re-run): Batch sanity check

fake_batch = torch.randn(8, 4)
output = quantum_layer(fake_batch)
print("Input shape:", fake_batch.shape)
print("Output shape:", output.shape)  # expect (8, 4)
print("Sample output:", output[0])

Input shape: torch.Size([8, 4])
Output shape: torch.Size([8, 4])
Sample output: tensor([-1.6406e-01,  7.8336e-02, -1.9827e-01,  5.1519e-05],
       grad_fn=<SelectBackward0>)


In [12]:
# Step 6: Define the full CNN + Quantum model
# Includes: shared backbone -> learnable per-feature scale + tanh bound
# -> quantum circuit -> Linear(4->1) output (Section 8.1)

class CNNQuantum(nn.Module):
    def __init__(self, quantum_layer):
        super().__init__()
        self.backbone = CNNBackbone()

        # Learnable per-feature scaling (4 params, no bias, no mixing)
        # Bounds output to (-pi, pi) via pi * tanh(x)
        self.scale = nn.Parameter(torch.ones(4))

        self.quantum_layer = quantum_layer

        self.output_layer = nn.Linear(4, 1)
        nn.init.kaiming_normal_(self.output_layer.weight, nonlinearity='relu')

    def forward(self, x):
        features = self.backbone(x)                        # (batch, 4)
        scaled = features * self.scale                       # learnable scale
        bounded = np.pi * torch.tanh(scaled)                  # bound to (-pi, pi)
        quantum_out = self.quantum_layer(bounded)              # (batch, 4)
        out = self.output_layer(quantum_out)                    # (batch, 1) raw logit
        return out

model = CNNQuantum(quantum_layer)
print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("\nTotal trainable parameters:", n_params)

# Breakdown check
scale_params = 4
quantum_params = 16
output_params = sum(p.numel() for p in model.output_layer.parameters())
print(f"Scale: {scale_params} | Quantum: {quantum_params} | Output: {output_params}")
print(f"Head total (should be 25): {scale_params + quantum_params + output_params}")

CNNQuantum(
  (backbone): CNNBackbone(
    (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1))
    (relu1): ReLU()
    (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1))
    (relu2): ReLU()
    (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (flatten): Flatten(start_dim=1, end_dim=-1)
    (bottleneck): Linear(in_features=800, out_features=4, bias=True)
  )
  (quantum_layer): <Quantum Torch Layer: func=quantum_circuit>
  (output_layer): Linear(in_features=4, out_features=1, bias=True)
)

Total trainable parameters: 8029
Scale: 4 | Quantum: 16 | Output: 5
Head total (should be 25): 25


In [13]:
# Step 7: Sanity check - run a real batch of MNIST images through the full model

sample_images, sample_labels = next(iter(train_loader))
output = model(sample_images)
print("Input shape:", sample_images.shape)
print("Output shape:", output.shape)  # expect (32, 1)
print("Sample logits:", output[:5].detach().numpy().flatten())

Input shape: torch.Size([32, 1, 28, 28])
Output shape: torch.Size([32, 1])
Sample logits: [ 1.0243584  -1.1892017   0.6931568  -1.3167887  -0.92975515]


In [14]:
# Step 8: Gradient sanity check (Section 10)
# Run one backward pass and check the quantum parameters' gradient norm.
# A norm near zero would indicate a "barren plateau" - the circuit not
# learning effectively. Must check this BEFORE committing to official runs.

criterion = nn.BCEWithLogitsLoss()

sample_images, sample_labels = next(iter(train_loader))
sample_labels = sample_labels.unsqueeze(1)

output = model(sample_images)
loss = criterion(output, sample_labels)
loss.backward()

print("Loss:", loss.item())
print()

# Check gradient norms for each parameter group
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        print(f"{name:30s} | shape {str(list(param.shape)):15s} | grad norm: {grad_norm:.6f}")
    else:
        print(f"{name:30s} | NO GRADIENT (problem!)")

Loss: 0.982683539390564

scale                          | shape [4]             | grad norm: 0.069009
backbone.conv1.weight          | shape [16, 1, 3, 3]   | grad norm: 0.516979
backbone.conv1.bias            | shape [16]            | grad norm: 0.216832
backbone.conv2.weight          | shape [32, 16, 3, 3]  | grad norm: 2.858238
backbone.conv2.bias            | shape [32]            | grad norm: 0.175609
backbone.bottleneck.weight     | shape [4, 800]        | grad norm: 9.239103
backbone.bottleneck.bias       | shape [4]             | grad norm: 0.145037
quantum_layer.weights          | shape [2, 4, 2]       | grad norm: 0.227761
output_layer.weight            | shape [1, 4]          | grad norm: 0.415977
output_layer.bias              | shape [1]             | grad norm: 0.068265


In [15]:
# Step 9: Timing check - confirm the quantum layer scales roughly
# LINEARLY with batch size, not exponentially (Section 14 requirement)
# A hidden per-sample loop would show up here as a huge time jump

import time

for test_batch_size in [8, 16, 32]:
    test_batch = torch.randn(test_batch_size, 4)
    start = time.time()
    _ = quantum_layer(test_batch)
    elapsed = time.time() - start
    print(f"Batch size {test_batch_size:3d} -> {elapsed*1000:.2f} ms")

print("\nIf times roughly double as batch size doubles (linear), we're good.")
print("If time explodes (e.g. 10x+ per doubling), that indicates a hidden")
print("per-sample loop bug despite the earlier shape fix.")

Batch size   8 -> 63.91 ms
Batch size  16 -> 28.66 ms
Batch size  32 -> 16.80 ms

If times roughly double as batch size doubles (linear), we're good.
If time explodes (e.g. 10x+ per doubling), that indicates a hidden
per-sample loop bug despite the earlier shape fix.


In [16]:
# Step 9b: Re-run the timing check to remove first-call "warm-up" bias
# (Step 9's decreasing pattern was likely PennyLane compiling on first call)

# Warm-up call (not timed)
_ = quantum_layer(torch.randn(32, 4))

print("Timing after warm-up:")
for test_batch_size in [8, 16, 32, 64]:
    test_batch = torch.randn(test_batch_size, 4)
    start = time.time()
    _ = quantum_layer(test_batch)
    elapsed = time.time() - start
    print(f"Batch size {test_batch_size:3d} -> {elapsed*1000:.2f} ms")

print("\nNow we should see roughly linear scaling with batch size.")

Timing after warm-up:
Batch size   8 -> 12.09 ms
Batch size  16 -> 13.30 ms
Batch size  32 -> 12.10 ms
Batch size  64 -> 12.35 ms

Now we should see roughly linear scaling with batch size.


In [17]:
# Step 10: Run a real 1-epoch training test on seed 42 - confirm the
# full pipeline trains without errors before scaling to 5-seed official runs

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        labels = labels.unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)
model = CNNQuantum(qml.qnn.TorchLayer(quantum_circuit, weight_shapes))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Starting 1 epoch training (this will take a few minutes - quantum simulation is slower)...")
start = time.time()
train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
elapsed = time.time() - start

print(f"\nEpoch complete in {elapsed:.1f} seconds")
print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f}")

Starting 1 epoch training (this will take a few minutes - quantum simulation is slower)...

Epoch complete in 13.5 seconds
Train loss: 0.5253 | Train acc: 0.7357


In [18]:
# Step 11: Run a few more epochs to see if it continues improving normally
# (not a full official run - just confirming healthy convergence behavior)

for epoch in range(2, 6):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    print(f"Epoch {epoch} | Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f}")

Epoch 2 | Train loss: 0.1659 | Train acc: 0.9967
Epoch 3 | Train loss: 0.0516 | Train acc: 0.9990
Epoch 4 | Train loss: 0.0245 | Train acc: 0.9997
Epoch 5 | Train loss: 0.0144 | Train acc: 1.0000


In [19]:
# Step 12: Pilot phase documentation sketch (Chunk 6) - UPDATED with 5-epoch convergence data

pilot_documentation = {
    "purpose": "Diagnose and fix problems in the CNN+Quantum model BEFORE committing to "
               "the full 5-seed official runs (Section 14). Pilot results are excluded "
               "from final statistical analysis.",

    "issues_found_and_fixed": [
        {
            "issue": "lightning.qubit backend failed during batched execution with "
                     "qml.qnn.TorchLayer (RuntimeError on reshape)",
            "fix": "Fell back to default.qubit, which correctly supports batch broadcasting. "
                   "Pre-approved fallback per Section 8.3.",
            "impact": "Runtime slightly slower than lightning.qubit would have been, "
                      "but correctness confirmed. Documented per protocol requirement."
        },
        {
            "issue": "CRITICAL BUG: circuit indexed inputs as inputs[i] (grabbing SAMPLE i "
                     "from the batch) instead of inputs[..., i] (grabbing FEATURE i across "
                     "the whole batch). Caused RuntimeError on any batch size > 1.",
            "fix": "Changed to inputs[..., i] - correct feature-wise indexing for batched execution.",
            "impact": "This is exactly the kind of hidden bug Section 14 warns about - would "
                      "have silently broken (or forced an incorrect per-sample workaround) "
                      "the official 5-seed runs if not caught here first."
        }
    ],

    "diagnostics_passed": {
        "gradient_health": "All parameters (including 16 quantum rotation weights) received "
                            "non-zero gradients on first backward pass. Quantum weight grad "
                            "norm: 0.227761 - healthy, no barren-plateau symptom observed.",
        "batch_scaling": "Confirmed near-flat timing (~12-13ms) across batch sizes 8-64 after "
                          "warm-up - confirms vectorized/batched execution via qml.qnn.TorchLayer, "
                          "NOT a hidden per-sample Python loop.",
        "backend_confirmed": "default.qubit (lightning.qubit fallback triggered and documented)",
        "single_epoch_training": "Completed successfully in 13.5 seconds, no errors.",
        "reproducibility": "Seed 42 used consistently, matches Section 14's requirement to use "
                            "the same first official seed across all pilot checks."
    },

    "architecture_confirmed": {
        "total_params": 8029,
        "head_params": 25,
        "breakdown": "4 (learnable scale) + 16 (quantum RY/RZ weights) + 5 (Linear 4->1 output) = 25",
        "matches_classical_control": True,
        "note": "Exact parameter-count match with Chunk 5's CNN+Classical Control (25 head "
                "params), as required by Section 8.1 for a fair comparison."
    },

    "early_training_behavior_5_epochs": {
        "epoch_1_train_acc": 0.7357,
        "epoch_1_train_loss": 0.5253,
        "epoch_2_train_acc": 0.9967,
        "epoch_2_train_loss": 0.1659,
        "epoch_3_train_acc": 0.9990,
        "epoch_3_train_loss": 0.0516,
        "epoch_4_train_acc": 0.9997,
        "epoch_4_train_loss": 0.0245,
        "epoch_5_train_acc": 1.0000,
        "epoch_5_train_loss": 0.0144,
        "note": "Quantum branch needed ~1 extra epoch to 'warm up' compared to classical "
                "models (which were already ~99% by epoch 1), but converges cleanly and "
                "smoothly afterward, reaching 100% train accuracy by epoch 5 with steadily "
                "decreasing loss. No signs of instability, plateauing, or divergence. "
                "This is a genuine, informative timing difference worth noting in the "
                "final write-up's Discussion (Section 8) - not a defect."
    },

    "decision": "Pilot confirms the CNN+Quantum pipeline is correct, trains without errors, "
                "has healthy gradients, executes in proper batched form, and converges cleanly "
                "over a handful of epochs (with a slightly delayed start vs classical models). "
                "PROTOCOL FROZEN - ready to proceed to Chunk 7/8's full 5-seed official runs, "
                "with NO further reactive tuning per Section 13's critical rule.",

    "batch_size_used": 32,
    "batch_size_fallback_needed": False,
    "learning_rate_group_fallback_needed": False
}

with open(f"{base}/documentation/06_Pilot_Phase_documentation_sketch.json", "w") as f:
    json.dump(pilot_documentation, f, indent=2)

print("Saved documentation sketch to documentation/06_Pilot_Phase_documentation_sketch.json")
print("\n--- CHUNK 6 (Pilot Phase) COMPLETE ---")

Saved documentation sketch to documentation/06_Pilot_Phase_documentation_sketch.json

--- CHUNK 6 (Pilot Phase) COMPLETE ---
